In [44]:
# function that constructs a trip based on a group of timestamps for a vehicle_id
def construct_trip(group):
    group_sorted = group.sort_values("timestep_time")
    first_row = group_sorted.iloc[0]
    last_row = group_sorted.iloc[-1]

    return pd.Series(
        {
            "start_x": first_row["vehicle_x"],
            "start_y": first_row["vehicle_y"],
            "destination_x": last_row["vehicle_x"],
            "destination_y": last_row["vehicle_y"],
            "manhattan_distance": abs(last_row["vehicle_x"] - first_row["vehicle_x"])
            + abs(last_row["vehicle_y"] - first_row["vehicle_y"]),
            "euclidean_distance": np.linalg.norm(
                first_row[["vehicle_x", "vehicle_y"]] - last_row[["vehicle_x", "vehicle_y"]]
            ),
            "hour_of_day": group["hour_of_day"].mode()[0],
            "vehicle_type_bus": first_row["vehicle_type_bus"],
            "vehicle_type_car": first_row["vehicle_type_car"],
            "vehicle_type_ldv": first_row["vehicle_type_ldv"],
            "vehicle_type_truck": first_row["vehicle_type_truck"],
            "trip_duration": last_row["timestep_time"] - first_row["timestep_time"],
        }
    )


# group by vehicle_id and construct the trips dataframe
trips_train = fcd_train.groupby("vehicle_id").apply(construct_trip, include_groups=False).reset_index()
trips_test = fcd_test.groupby("vehicle_id").apply(construct_trip, include_groups=False).reset_index()